In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import tensorflow as tf
#from tensorflow.keras import layers, models, losses
#from tensorflow.keras.callbacks import ModelCheckpoint
from keras import layers, models, losses, regularizers
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

I0000 00:00:1780658165.225208   22756 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print("GPU Trovate:", len(tf.config.list_physical_devices('GPU')))
for gpu in tf.config.list_physical_devices('GPU'):
    print("Nome:", gpu.name)

# 1. SETUP MEMORIA: DEVE ESSERE LA PRIMA COSA IN ASSOLUTO
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memoria GPU configurata in modalità dinamica.")
    except RuntimeError as e:
        print("Errore GPU:", e)

# 2. ESORCISMO DELLA RAM (Uccide i vecchi modelli in memoria)
tf.keras.backend.clear_session()

GPU Trovate: 1
Nome: /physical_device:GPU:0
Memoria GPU configurata in modalità dinamica.


# [LOG] Model Versioning

## Version 1: Tiny-Baseline (Obsolete)
**Data:** 15/05/2026
**Fase:** Upper-Bound Baseline (Test di fattibilità hardware)

### Architettura:
- **Input:** (1, 120, 18) -> [H, W, Channels]
- **Feature Extraction:** - Conv2D (16 filtri, kernel 1x5) + MaxPooling (1x2)
    - SeparableConv2D (32 filtri, kernel 1x3) + MaxPooling (1x2)
- **Output Heads:** - `coords_head`: Dense(8) [Linear] -> X, Y per 4 persone.
    - `mask_head`: Dense(4) [Sigmoid] -> Presenza per 4 persone.

### Statistiche:
- **Parametri Totali:** ~3,500
- **Peso Modello (Float32):** 13.67 KB
- **Peso Stimato (INT8 Quantized):** ~3.5 KB
- **Performance (Epoca 15):** - `val_loss`: 3.79
    - `val_coords_loss`: 3.35 (Errore spaziale medio ~1.83m)
    - `val_mask_loss`: 0.88

## Version 2: Capacità Espansa (Obsolete)
* **Performance:** Errore medio ~1.69m. 
* **Note:** La rete ha smesso di imparare dopo 37 epoche. Mancanza di regolarizzazione (Dropout) e LR fisso.

## Version 3: Architettura "Romana" (Heavy + Callbacks)
**Data:** 31/05/2026
**Fase:** Ottimizzazione Avanzata
* **Architettura:** 12 Layer. Doppie Conv2D(32) -> Doppie SepConv2D(64) -> Conv2D(128) -> Dense(128) + Dropout(0.3) -> Dense(64).
* **Data Pipeline:** `alpha=0.20` (EMA decluttering veloce, come da specifiche). Split dataset con casi complessi (3/4 persone) nel Training. `Batch_Size=8`.
* **Training Hacks:** - `ReduceLROnPlateau`: dimezza il learning rate se la loss si blocca.
    - `EarlyStopping`: ferma l'addestramento se non migliora per 10 epoche e ricarica i pesi migliori.
    - Metrica `RootMeanSquaredError`: legge l'errore spaziale direttamente in Metri.
* **Performance Spaziale:** L'errore medio sulle coordinate è sceso al minimo storico di **~1.56m**.

## [LA SERIE DEI DISASTRI TEORICI: V4 - V6]
*I modelli seguenti rappresentano tentativi falliti di risolvere il problema dell'assegnazione spaziale (Permutation Invariance) e di ignorare i "fantasmi" (persone non presenti). Hanno portato a una serie di Mode Collapse a causa di bug matematici e concettuali.*

## Version 4: Architettura "Imperiale" (Obsolete - Disastroso / Mode Collapse)

### Modifiche Apportate (La Teoria):
- **Spatial Sorting:** Ordinamento forzato dei target da sinistra a destra sull'asse X nel DataGenerator per fornire una regola fissa alla rete (aggirare la *permutation invariance*). **GRAVE ERRORE!!**
- **True Masked MSE:** Modifica della Loss function per moltiplicare l'errore per 0 quando la persona non è presente, smettendo di penalizzare la rete per i "fantasmi".
- **Bilanciamento Loss:** Il peso della `mask_head` è stato portato da 0.5 a 5.0 per costringere l'ottimizzatore a prestare attenzione alla presenza.
- **Custom Metric:** Creata `true_masked_rmse_metres` per calcolare correttamente l'errore in metri gestendo gli array concatenati a 12 valori (8 coords + 4 maschere).

### Statistiche e Limiti Hardware:
- Aumentare indiscriminatamente i filtri a 128 e i layer Densi a 256 ha causato un **esplosione della memoria Flash stimata a 927.42 KB**, superando il limite tassativo di 800 KB dell'ESP32. 

### Il Disastro (Performance):
- Rispetto al modello V3 (e allo split di Davide), **la resa visiva è pessima**. 
- **Sintomo:** Nel visualizzatore, le predizioni (le X rosse) non inseguono minimamente i bersagli. Rimangono immobili, raggruppate e appiccicate in un singolo punto nell'angolo in basso a sinistra della stanza.
- **Causa:** Il layer `GlobalAveragePooling2D`. Calcolando la media matematica su tutta l'ultima mappa di estrazione, ha letteralmente distrutto ogni informazione geometrica. La rete è diventata **cieca**. Non sapendo *dove* guardare, ha applicato un "Mode Collapse": ha imparato a sparare tutte le previsioni nel punto medio statistico per subire la minor penalità possibile dalla Loss.

## Version 5: Architettura "Occhiali" (Obsolete - Mode Collapse Persistente)
* **Modifica Architetturale:** Sostituito il `GlobalAveragePooling2D` con il layer `Flatten()` per ridare alla rete la "vista" geometrica. Usati `MaxPooling2D` aggressivi per mantenere la Flash stimata sotto gli 800 KB (scesa a ~223 KB).
* **Risultato:** Fallimento. Le predizioni continuano a non seguire i bersagli.
* **Diagnosi (Il vero colpevole):** Il problema non era solo il Pooling, ma lo **Spatial Sorting** introdotto nella V4. Poiché le persone si incrociano nella stanza, ordinare le coordinate sull'asse X frame per frame causava il "teletrasporto" dei target da un output all'altro. La rete, non avendo memoria temporale (LSTM), non riusciva a gestire questi sbalzi di gradiente e collassava statisticamente.

## Version 6: Architettura "Tracciante Naturale" (Obsolete - Bug Matematico)
* **Modifica:** Rimosso lo Spatial Sorting, ripristinato l'ordine naturale del dataset. Fixato il bug `axis=1` nella Loss.
* **Risultato:** Fallimento totale e crollo della Binary Accuracy (~50%). Errore fisso a 1.54m misurato magicamente al centro della stanza.
* **Causa 1 (Bug Keras):** Nella funzione custom *True Masked MSE*, mancava il parametro `axis=1` nell'istruzione `tf.reduce_sum()`. Questo fondeva l'errore di un intero batch in un singolo scalare, distruggendo completamente i gradienti frame-per-frame e lobotomizzando la rete.
* **Causa 2:** Problema dell'Assegnazione. Senza ordinamento spaziale, la rete (che non ha memoria temporale LSTM) non sa in quale delle 4 teste di output piazzare le coordinate di una persona in un singolo frame isolato, finendo per sparare al centro statistico per minimizzare la penalità.

## Version 7: Architettura "Ungara" (Breakthrough)
**Fase:** Risoluzione del Problema di Assegnazione
* **Modifica Teorica:** Fusa l'architettura in un'unica testa di output da 12 valori. Introdotta la **Total Hungarian Loss** (Permutation Invariant Training): la rete ora calcola l'errore per tutte le 24 permutazioni possibili e impara solo dall'incrocio geometricamente perfetto, eliminando il Mode Collapse.
* **Risultato:** Le predizioni si sbloccano dal centro della stanza e iniziano a inseguire fisicamente i target reali in movimento.
* **Criticità rilevate (Bug di Misurazione e Metodo):**
    1. **Bug Metrica Spaziale:** Il calcolo dell'errore (0.95m riportati) divideva per il numero di assi, distorcendo la trigonometria (il vero errore euclideo era ~1.34m).
    2. **Bug Metrica Maschera:** Keras misurava l'accuratezza binaria (riportata al 44%) su tutti i 12 output, mescolando coordinate e probabilità.
    3. **Data Split Sbilanciato:** Il set di Validazione ometteva del tutto gli scenari con 2 persone, minando la validità scientifica del test.!!!!

## Version 8: Architettura "Corazzata" (The Truth Fix)

**Fase**: Rigore Matematico e Hard-Limit Hardware

* **Data Split Stratificato:** Copertura totale in Validation (0, 1, 2, 3 e 4 soggetti) per testare l'algoritmo su tutti gli scenari.

* **Architettura Espansa:** Filtri raddoppiati (fino a 128) e livello Dense portato a 256. Saturazione consapevole della Flash (~683 KB) per estrarre il massimo dettaglio dai radar. Unione definita delle teste (Concatenate) per sincronizzare il Permutation Invariant Training.

* **Fix Matematici:** Metrica Euclidea reale (hungarian_rmse_euclidean_metres) e metrica di accuratezza maschere purificata (hungarian_mask_acc).

* **Performance (Epoca 23 - Golden):**

    *val_hungarian_rmse_metres: ~1.07m (Distanza Reale Fisica).

    *val_hungarian_mask_acc: ~77% (Capacità di distinguere fantasmi da persone reali).

* **Risultato Visivo:** Le predizioni inseguono fedelmente i bersagli garantendo il requisito F1 Score (True Positive <= 1.0m). Permane un fisiologico sfarfallio sui "fantasmi" dovuto all'assenza di memoria temporale (LSTM).

## Version 8.1: Architettura "Corazzata" (Full-Batch Survival)
**Data:** 05/06/2026
**Fase:** Sblocco Hardware GPU, Diagnosi e Record Full-Batch

### Requisiti Hardware (ESP32-S3 - FLOAT32):
- **Memoria FLASH Stimata:** 683.92 KB (✅ Limite: 800 KB).
- **Memoria SRAM Stimata (Activation Arena):** ~8.44 KB (✅ Limite: 400 KB).

### Modifiche Apportate (Ingegneria di Sistema):
- **Accensione Reattore GPU:** Risolto il blocco driver passando dall'addestramento su CPU a quello su NVIDIA GTX 1650 (4GB VRAM) tramite l'installazione nativa dei toolkit CUDA e cuDNN via Conda.
- **cuDNN Autotuner Bypass:** Disattivato l'autotuning di TensorFlow (`TF_CUDNN_USE_AUTOTUNE = 0`). L'Autotuner tentava un collaudo di algoritmi convoluzionali che richiedeva oltre 1GB di workspace VRAM, portando la scheda al crash istantaneo. Disattivandolo, l'addestramento procede in modo deterministico e sicuro.
- **Memoria Dinamica e OOM Fix:** Inserito esorcismo della RAM (`clear_session()`) per distruggere i grafi orfani. Impostato il limite tassativo `BATCH_SIZE = 1` nel `EEAIDataGenerator` per evitare il crash per "Indigestione" da allocazione VRAM singola (>2.5 GB).

### Performance:
- **`val_hungarian_rmse_metres`:** **0.8380m** (Nuovo record assoluto! Muro dell'1.0m ufficialmente abbattuto).
- **`val_hungarian_mask_acc`:** **79.23%** (Ottima reiezione dei fantasmi).
- **Tempo di addestramento:** ~15 secondi per Epoca.

### Il Limite Concettuale Scoperto:
- Sebbene la rete abbia performato benissimo, si è scoperta una grave anomalia architetturale nel DataGenerator legata al `BATCH_SIZE = 1`.
- Keras interpreta `1` non come *1 frame*, ma come *1 file intero* (circa 15.000 frame cronologici).
- **L'Errore Matematico:** Questo costringe la rete a eseguire un **Full-Batch Gradient Descent**. La rete fa la media degli errori di tutte le 15.000 istantanee e aggiorna i suoi pesi **una sola volta** per ogni file. Con 18 file di Train, la rete ha compiuto appena 18 passi di discesa del gradiente per ogni Epoca.

### Verdetto:
Questa versione rappresenta la **Baseline Assoluta (Strong)** dell'infrastruttura. Aver raggiunto 0.83m con soli 18 aggiornamenti per epoca dimostra che l'architettura V8 "Corazzata" e la funzione di Loss Ungherese sono perfette. Tuttavia, segna la fine dell'uso del `tf.keras.utils.Sequence` per questo dataset: la rete è pronta per il VERO Machine Learning. Passando all'infrastruttura in RAM (Mini-Batch da 32), la rete passerà da 18 aggiornamenti a oltre 6.000 aggiornamenti per epoca, aprendo le porte alla micro-cesellatura dei pesi.

# [GUIDA] Gerarchia del Fine-Tuning per Edge AI (ESP32-S3)

Nel TinyML non possiamo ingrandire la rete a caso, perché siamo limitati da 400KB di RAM e dalla latenza. Le modifiche seguono un ordine di priorità basato sul **Costo Hardware**.

### Livello 1: Costo Hardware ZERO (Modifiche di Addestramento)
Questi parametri non alterano il peso finale del file `.tflite`. Si provano per primi.
* **1. Epoche (`epochs`):** * *Cos'è:* Il tempo di studio. Quante volte la rete vede l'intero dataset.
    * *Quando usarlo:* Se la `val_loss` sta scendendo ma l'addestramento finisce troppo presto (Underfitting).
    * *Effetto:* Permette alla rete di continuare a correggere gli errori.
* **2. Learning Rate (`lr`):**
    * *Cos'è:* La "lunghezza del passo" durante la discesa del gradiente.
    * *Quando usarlo:* Se la Loss salta su e giù in modo impazzito (LR troppo alto) o se non scende per niente fin dall'inizio (LR troppo basso).
    * *Effetto:* Rende l'apprendimento più stabile o più aggressivo.

### Livello 2: Costo Hardware BASSO (Capacità / Larghezza)
* **3. Numero di Filtri (es. da 32 a 64):**
    * *Quando usarlo:* Se la rete è troppo "stupida" per capire le dinamiche della stanza e la Loss si blocca su valori alti (come nella nostra V1).
    * *Effetto:* Aumenta i parametri (Flash) e leggermente la RAM. Dà alla rete più "neuroni" per capire la trigonometria.

### Livello 3: Costo Hardware ALTO (Profondità / Latenza)
* **4. Aggiungere Layer (es. una terza Conv2D):**
    * *Cos'è:* Aggiungere step sequenziali al modello.
    * *Quando usarlo:* Solo se la rete larga non basta per estrarre concetti complessi.
    * *Effetto:* Aumenta drasticamente le operazioni matematiche (MACs). **Aumenta la latenza:** l'ESP32 ci metterà molto più tempo a calcolare ogni singolo frame. Usare con estrema cautela.

In [ ]:
# ==============================================================================
# DATA ENGINE V6 (Output FIXATO per il modello V8, Split Originale)
# ==============================================================================
class EEAIDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, file_paths, batch_size=8, alpha=0.20, is_training=True):
        self.file_paths = file_paths
        self.batch_size = batch_size
        self.alpha = alpha
        self.is_training = is_training
        if self.is_training:
            np.random.shuffle(self.file_paths)

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_files = self.file_paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch, y_coords_mask_batch, y_mask_batch = [], [], []

        for file_path in batch_files:
            data = np.load(file_path)
            raw_iq = data['radar_cir_iq']   
            people_xy = data['people_xy']   
            people_mask = data['people_mask'] 
            T = raw_iq.shape[0]             
            
            mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
            mag_reshaped = mag.reshape(T, 1, 120, 18) 
            
            bg = np.copy(mag_reshaped[0])
            decluttered = np.zeros_like(mag_reshaped)
            
            for t in range(T):
                # 1. EMA Decluttering
                bg = self.alpha * mag_reshaped[t] + (1 - self.alpha) * bg
                decluttered[t] = np.abs(mag_reshaped[t] - bg)
            
            # 2. RIMOZIONE SORTING
            flat_coords = people_xy.reshape(T, 8)
            
            # 3. Y_combined ha già 12 valori (8 coords + 4 mask)
            combined_target = np.concatenate([flat_coords, people_mask], axis=1)

            X_batch.append(decluttered)
            y_coords_mask_batch.append(combined_target)
            y_mask_batch.append(people_mask)

        X = np.concatenate(X_batch, axis=0).astype(np.float32)       
        
        # Questa variabile contiene già tutti i 12 target necessari per la V8
        Y_combined = np.concatenate(y_coords_mask_batch, axis=0).astype(np.float32) 
        
        # LA MODIFICA È QUI: Restituiamo solo X e il target combinato
        return X, Y_combined

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.file_paths)

# ==============================================================================
# IL TUO SPLIT ORIGINALE
# ==============================================================================
#train_indices = [22, 0, 1, 2, 3, 10, 14, 18, 19, 21, 8, 9, 12, 5, 6, 4, 7, 13]
#val_indices = [23, 16, 11, 17, 15, 20]

# ==============================================================================
# SPLIT STRATIFICATO RIGOROSO (Copertura classi: 0,1,2,3,4 soggetti)
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] # 1 per categoria + un 4 extra
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

BATCH_SIZE = 1

train_gen = EEAIDataGenerator(train_files, batch_size=BATCH_SIZE, alpha=0.20, is_training=True)
val_gen = EEAIDataGenerator(val_files, batch_size=BATCH_SIZE, alpha=0.20, is_training=False)

print(f"Motore pronto: {len(train_files)} file di Train, {len(val_files)} file di Validation.")

Motore pronto: 18 file di Train, 6 file di Validation.


In [4]:
import itertools

PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (2.0 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (2.0 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

I0000 00:00:1780658167.683323   22756 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2547 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [5]:
def embedded_summary(model, input_shape=(1, 120, 18)):
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
    max_layer_ram_kb = 0
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    print("============================================")
    print("   REPORT REQUISITI ESP32-S3 (FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite: 400 KB)")
    print("============================================\n")

In [6]:
# ==============================================================================
# 2. ARCHITETTURA EEAI-NET V8 "CORAZZATA" 
# ==============================================================================
def build_eeai_model_v8_corazzata(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(64, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(64, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 3))(x) 
    
    x = layers.Flatten(name="flatten_spatial_map")(x) 
    
    x = layers.Dense(256, activation='relu', name="features_deep_1")(x)
    x = layers.Dropout(0.3, name="drop_features")(x) 
    common_feat = layers.Dense(128, activation='relu', name="features_deep_2")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    # UNIONE DELLE TESTE PER L'HUNGARIAN MATCHING
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V8_Corazzata")

model_v8_final = build_eeai_model_v8_corazzata()

if 'embedded_summary' in globals():
    embedded_summary(model_v8_final)  # Torneremo ai nostri sicuri ~684 KB!

# ==============================================================================
# 3. FINE TUNING: OTTIMIZZATORE E CALLBACKS
# ==============================================================================
# Learning rate dimezzato in partenza (0.0005) per una discesa chirurgica
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model_v8_final.compile(
    optimizer=optimizer,
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

# Pazienza aumentata (5 e 15) per cuocere la rete a fuoco lento
checkpoint_v8 = ModelCheckpoint("eeai_best_model_v8.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)

print("\n--- INIZIO ADDESTRAMENTO V8 CORAZZATA (FINE-TUNING STRATIFICATO) ---")
history_v8_final = model_v8_final.fit(
    train_gen,                
    validation_data=val_gen,  
    epochs=100,
    callbacks=[checkpoint_v8, reduce_lr, early_stop], 
    verbose=1
)

   REPORT REQUISITI ESP32-S3 (FLOAT32)   
 Memoria FLASH stimata : 683.92 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~8.44 KB (Limite: 400 KB)


--- INIZIO ADDESTRAMENTO V8 CORAZZATA (FINE-TUNING STRATIFICATO) ---


/home/marco/yes/envs/edge_ai_env/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100


I0000 00:00:1780658169.595967   22756 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1780658171.558183   22821 service.cc:153] XLA service 0x7ff1d0038c90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780658171.558206   22821 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1650, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.10.1)
I0000 00:00:1780658171.617121   22821 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780658171.943777   22821 cuda_dnn.cc:461] Loaded cuDNN version 91001
I0000 00:00:1780658171.998076   22821 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3877__.63
W0000 00:00:1780658172.565159   22894 hlo_rematerialization.cc:3204] Can't reduce memory use below 2.60GiB (2796141671 bytes) by rematerialization; only reduced to

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 579ms/step - hungarian_mask_acc: 0.5105 - hungarian_rmse_metres: 9.7319 - loss: 345.5362

I0000 00:00:1780658245.757360   22823 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_4507__.24
W0000 00:00:1780658246.065811   23089 hlo_rematerialization.cc:3204] Can't reduce memory use below 2.70GiB (2904489376 bytes) by rematerialization; only reduced to 29.86GiB (32064011552 bytes), down from 29.86GiB (32064011552 bytes) originally
W0000 00:00:1780658256.231741   22823 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 29.75GiB (rounded to 31948812032)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1780658256.231783   22823 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1780658256.231786   22823 bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 43, Chunks in use: 43. 10.8KiB allocated for chunks. 10.8KiB in use in bin. 


Epoch 1: val_loss improved from None to 13.47507, saving model to eeai_best_model_v8.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - hungarian_mask_acc: 0.5287 - hungarian_rmse_metres: 7.2189 - loss: 184.0468 - val_hungarian_mask_acc: 0.5422 - val_hungarian_rmse_metres: 2.5376 - val_loss: 13.4751 - learning_rate: 5.0000e-04
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 641ms/step - hungarian_mask_acc: 0.5473 - hungarian_rmse_metres: 2.7623 - loss: 14.9026
Epoch 2: val_loss improved from 13.47507 to 8.05556, saving model to eeai_best_model_v8.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 16s 871ms/step - hungarian_mask_acc: 0.5782 - hungarian_rmse_metres: 2.7605 - loss: 13.8076 - val_hungarian_mask_acc: 0.5813 - val_hungarian_rmse_metres: 2.0031 - val_loss: 8.0556 - learning_rate: 5.0000e-04
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 648ms/step - hungarian_mask_acc: 0.5513 - hungarian_rmse_metres: 2.5552 - loss: 10.1674
Epoch 3: val_loss improved from 8.05556 to 6.64756, saving model to eeai_best_model_

### Come leggere le Metriche della nostra Hungarian EEAI-Net (V7)

Con l'introduzione dell'**Hungarian Matching** (Permutation Invariant Training), le nostre due teste sono state fuse per calcolare un unico costo globale. Ecco i 4 valori fondamentali che vedrai scorrere sullo schermo e come interpretarli:

1. **hungarian_rmse_metres (L'Errore Spaziale Reale)**
È il traduttore fisico. Indica la distanza media in metri tra le tue predizioni (le X rosse) e le persone reali (i pallini verdi), misurata **dopo** che l'algoritmo ha trovato l'incrocio matematico perfetto. 
*Esempio:* Se vale `1.54`, stai sbagliando in media di 1 metro e mezzo. Più questo numero scende verso lo zero, più le X rosse "inseguiranno" fedelmente i bersagli veri, ignorando i fantasmi.

2. **loss (Il Voto Negativo Globale - Il motore dei gradienti)**
È il "costo" complessivo che la rete usa per correggere i propri errori sui dati di Addestramento. Unisce due punizioni: 
- L'errore balistico di posizione (MSE).
- La penalità se allucina "fantasmi" in slot vuoti (Binary Crossentropy moltiplicata per `2.5`). 
La loss viene calcolata *solo ed esclusivamente* sulla permutazione migliore delle 24 possibili. La rete cerca disperatamente di abbassare questo numero aggiornando i suoi pesi convoluzionali.

3. **val_loss (La Validation Loss - Il Re assoluto dell'addestramento)**
È lo stesso identico calcolo globale della `loss`, ma applicato ai dati che la rete **non ha mai visto** (l'esame di maturità a libro chiuso, es. le finestre 11 e 15). 
*Attenzione:* Questo è il numero più importante di tutti. L'`EarlyStopping` e il `ModelCheckpoint` guardano *esclusivamente* la `val_loss` per capire se il modello sta generalizzando la fisica del radar o se si sta solo imparando a memoria il training set (Overfitting). Finché scende, sei sulla strada giusta.

4. **val_hungarian_rmse_metres (La Prestazione Operativa - Il numero per la Tesi)**
È l'errore spaziale in metri misurato sui dati sconosciuti di validazione. Questo è il dato ufficiale che certificherà la bontà del tuo progetto. Nel tuo report o nella tesi scriverai: *"Il modello ha dimostrato un errore operativo reale di X metri su scenari mai visti durante l'addestramento"*.

Epoch 92/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - hungarian_mask_acc: 0.6643 - hungarian_rmse_metres: 1.0836 - loss: 2.8259 
Epoch 92: val_loss improved from 2.85260 to 2.82035, saving model to eeai_best_model_v8.keras


In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.1 (Fix Output 12 Dimensioni V8)
# ==============================================================================

file_target = "dataset/data/window_000011.npz"

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V8)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    print("Caricamento dei pesi migliori dal file .keras ...")
    
    # Caricamento del modello 
    model_v8_best = load_model(
        "eeai_best_model_v8.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_v8_best.predict(decluttered, verbose=0)
    
    # ==========================================
    # IL FIX E' QUI: Slicing corretto per la V8
    # ==========================================
    # preds ha dimensione [T, 12]. 
    # Prendiamo le prime 8 colonne per le coordinate, e le ultime 4 per le maschere.
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # Titolo aggiornato
            ax.set_title(f"Radar V9 | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (V8)...
Caricamento dei pesi migliori dal file .keras ...


I0000 00:00:1780659794.897386   22821 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_28023__.9
I0000 00:00:1780659795.786334   22822 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_28613__.9


Dati pronti! Inizializzazione Radar...
